# Tutorial 6: Your First Investigation

**Prerequisites:** T00–T05 (all foundational tutorials)

**What you'll learn:**
- How to conduct a complete mechanistic interpretability investigation
- The workflow: observe → hypothesize → intervene → verify
- How to use the tools from T02–T05 together
- How to write up your findings

---

## The Investigation: Repeated Token Detection

We'll investigate a simple behavior: **does the model "know" when a token has appeared before in the sequence?**

Even with random weights, attention patterns can create interesting structure. With our small model, we'll trace the complete information flow from input to output.

### The Scientific Method for Mechinterp

1. **Observe**: Run the model, capture activations, look for interesting patterns
2. **Hypothesize**: "I think component X is responsible for behavior Y"
3. **Intervene**: Ablate or patch component X and check if Y changes
4. **Verify**: Confirm the hypothesis holds across multiple examples

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=2, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## Step 1: Observe — Compare Repeated vs. Non-Repeated Tokens

In [ ]:
# Create two inputs: one with a repeated token, one without
tokens_with_repeat = jnp.array([10, 20, 30, 20, 40])  # Token 20 repeats
tokens_no_repeat   = jnp.array([10, 20, 30, 50, 40])  # No repeats

logits_repeat, cache_repeat = model.run_with_cache(tokens_with_repeat)
logits_no_repeat, cache_no_repeat = model.run_with_cache(tokens_no_repeat)

# Compare the model's output at position 3 (where the repeat happens)
print('Position 3 comparison:')
print(f'  With repeat (token 20 seen before):    top prediction = '
      f'token {int(jnp.argmax(logits_repeat[3]))}')
print(f'  Without repeat (token 50 is new):      top prediction = '
      f'token {int(jnp.argmax(logits_no_repeat[3]))}')

# How different are the logits?
diff = float(jnp.linalg.norm(logits_repeat[3] - logits_no_repeat[3]))
print(f'\n  Logit difference at position 3: {diff:.4f}')
print(f'  (This tells us the model treats repeated tokens differently)')

In [ ]:
# Step 1b: Look at attention patterns — does anything attend to the first occurrence?
print('Attention from position 3 to other positions:')
print('(In the repeated case, position 1 has the same token as position 3)')
print()

for layer in range(cfg.n_layers):
    patterns = cache_repeat[f'blocks.{layer}.attn.hook_pattern']
    for head in range(cfg.n_heads):
        p = np.array(patterns[head, 3, :4])  # Query pos 3, all key positions
        # Highlight attention to position 1 (the first occurrence of token 20)
        marker = ' <-- repeat!' if p[1] > 0.3 else ''
        print(f'  L{layer}H{head}: [{"  ".join([f"{v:.2f}" for v in p])}]{marker}')

## Step 2: Hypothesize

Looking at the attention patterns, let's identify which heads attend most to the position where the repeated token first appeared. These are candidates for "duplicate token detection" heads.

**Hypothesis**: The head(s) that attend most to position 1 from position 3 are responsible for the different behavior on repeated tokens.

In [ ]:
# Find the head that attends most to the repeat position
best_head = None
best_attn = 0.0

for layer in range(cfg.n_layers):
    patterns = cache_repeat[f'blocks.{layer}.attn.hook_pattern']
    for head in range(cfg.n_heads):
        attn_to_repeat = float(patterns[head, 3, 1])  # pos 3 attending to pos 1
        if attn_to_repeat > best_attn:
            best_attn = attn_to_repeat
            best_head = (layer, head)

print(f'Hypothesis: L{best_head[0]}H{best_head[1]} is a duplicate-token detector')
print(f'  (It attends {best_attn:.2%} to the first occurrence of the repeated token)')

## Step 3: Intervene — Test the Hypothesis

If our hypothesis is correct, ablating this head should eliminate (or reduce) the difference between the repeated and non-repeated cases.

In [ ]:
# Measure the baseline difference
metric_pos = 3
baseline_diff = float(jnp.linalg.norm(
    logits_repeat[metric_pos] - logits_no_repeat[metric_pos]
))
print(f'Baseline logit difference at position {metric_pos}: {baseline_diff:.4f}')

# Ablate the candidate head and re-measure
def ablate_target(z, name, head=best_head[1]):
    return z.at[:, head, :].set(0.0)

hooks = {f'blocks.{best_head[0]}.attn.hook_z': ablate_target}

logits_repeat_abl = model.run_with_hooks(tokens_with_repeat, fwd_hooks=hooks)
logits_no_repeat_abl = model.run_with_hooks(tokens_no_repeat, fwd_hooks=hooks)

ablated_diff = float(jnp.linalg.norm(
    logits_repeat_abl[metric_pos] - logits_no_repeat_abl[metric_pos]
))
print(f'Ablated logit difference:  {ablated_diff:.4f}')

reduction = (baseline_diff - ablated_diff) / baseline_diff * 100
print(f'\nDifference reduced by {reduction:.1f}%')
if reduction > 30:
    print(f'Strong evidence: L{best_head[0]}H{best_head[1]} contributes significantly '
          f'to repeat-token processing!')
else:
    print(f'Weak evidence. The effect may be distributed across multiple heads.')

## Step 3b: Activation Patching — Where Does the Signal Live?

Let's use activation patching to find which components carry the "this token is repeated" signal.

In [ ]:
# Activation patching: swap activations from the non-repeat run into the repeat run
# If the difference decreases, that component carries the "repeat" signal

print('Activation patching results:')
print('(Swap each component from non-repeat into repeat run, measure remaining diff)\n')

for layer in range(cfg.n_layers):
    for component in ['hook_attn_out', 'hook_mlp_out']:
        hook_name = f'blocks.{layer}.{component}'
        
        # Get the activation from the non-repeat run
        no_repeat_act = cache_no_repeat[hook_name]
        
        # Patch it into the repeat run
        hooks = {hook_name: lambda x, n, act=no_repeat_act: act}
        logits_patched = model.run_with_hooks(tokens_with_repeat, fwd_hooks=hooks)
        
        patched_diff = float(jnp.linalg.norm(
            logits_patched[metric_pos] - logits_no_repeat[metric_pos]
        ))
        
        # If patching makes the repeat run look like the non-repeat run,
        # this component carries the repeat signal
        recovery = (baseline_diff - patched_diff) / baseline_diff * 100
        name = 'Attn' if 'attn' in component else 'MLP'
        bar = '#' * int(abs(recovery) / 5)
        print(f'  Layer {layer} {name}: {recovery:+.1f}% recovery {bar}')

print(f'\nHigher recovery = this component carries more of the "repeat" signal.')

## Step 4: Verify — Test Across Multiple Examples

In [ ]:
# Generate several repeat/non-repeat pairs and check consistency
rng = jax.random.PRNGKey(123)

print('Verification across 5 different token patterns:')
print(f'(Comparing attention of L{best_head[0]}H{best_head[1]} '
      f'to the repeated position)\n')

for trial in range(5):
    rng, k1, k2 = jax.random.split(rng, 3)
    base_tokens = jax.random.randint(k1, (5,), 1, cfg.d_vocab)
    
    # Create repeat version: copy token at position 1 to position 3
    repeat_tokens = base_tokens.at[3].set(base_tokens[1])
    
    _, c = model.run_with_cache(repeat_tokens)
    pattern = c[f'blocks.{best_head[0]}.attn.hook_pattern']
    attn = float(pattern[best_head[1], 3, 1])
    
    tokens_str = list(np.array(repeat_tokens))
    print(f'  Trial {trial}: tokens={tokens_str}, '
          f'attn[3->1] = {attn:.3f}')

print(f'\nConsistent high attention to the repeated position would confirm')
print(f'our hypothesis. With random weights, patterns will be noisy.')

## Step 5: Write Up the Investigation

Every mechinterp investigation should be documented with:

1. **Behavior studied**: What does the model do?
2. **Methodology**: What tools did you use?
3. **Findings**: What components are responsible?
4. **Evidence**: What interventions support this?
5. **Limitations**: What questions remain?

Here's an example write-up:

In [ ]:
print("""
INVESTIGATION: Repeated Token Processing
=========================================

Behavior: The model produces different outputs at position 3 depending
on whether the token at position 3 has appeared earlier in the sequence.

Methodology:
  1. Compared activations on matched pairs (repeat vs. non-repeat)
  2. Inspected attention patterns to find candidate heads
  3. Used head ablation to test causal importance
  4. Used activation patching to localize the signal

""")

print(f"Findings:")
print(f"  - L{best_head[0]}H{best_head[1]} attends {best_attn:.1%} "
      f"to the first occurrence of a repeated token")
print(f"  - Ablating this head reduces the repeat/non-repeat difference "
      f"by {reduction:.1f}%")
print(f"")
print(f"Limitations:")
print(f"  - Small model with random weights (not trained)")
print(f"  - Limited to 5 token sequences")
print(f"  - The effect may be distributed across multiple heads")
print(f"")
print(f"Next steps:")
print(f"  - Test with a trained model (GPT-2)")
print(f"  - Look for induction heads that compose with this head")
print(f"  - Study longer sequences with more complex patterns")

## Key Takeaways

You now know the **complete mechinterp workflow**:

1. **Observe**: `run_with_cache()` → inspect attention patterns, activations, logits
2. **Hypothesize**: Identify candidate components responsible for a behavior
3. **Intervene**: `run_with_hooks()` with ablation or patching to test causality
4. **Verify**: Test across multiple examples to confirm generality

The tools you've learned in T00–T06 are sufficient to conduct real research:
- T02: Hooking and caching to observe the model
- T03: Residual stream decomposition to attribute predictions
- T04: Attention head analysis (QK/OV circuits, ablation)
- T05: MLP/neuron analysis (activation patterns, logit effects)

### What's Next

The remaining tutorials cover **specific techniques** that build on this foundation:

| Tutorial | Technique | What it adds |
|----------|-----------|-------------|
| **T07** | Logit Lens | Watch predictions form layer by layer |
| **T08** | Activation Patching | Systematic causal hypothesis testing |
| **T09** | Circuit Discovery | Find the minimal circuit for a behavior |
| **T10** | Sparse Autoencoders | Disentangle superposed features |
| **T11** | Advanced Techniques | Steering, probing, and beyond |

**Next: [T07 — The Logit Lens](T07_the_logit_lens.ipynb)** — A powerful technique for watching the model's prediction form across layers.